In [ ]:
# ============================================================
# 0. 라이브러리
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import re

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer


# ============================================================
# 1. 데이터 경로 설정
# ============================================================

DATA_DIR = Path(
    "/Users/leeseungbo/Desktop/APTE_교통사고_데이터/DATA/2011~2019_고속도로 교통사고 데이터"
)

print("DATA_DIR 존재 여부:", DATA_DIR.exists())
print("DATA_DIR 폴더 여부:", DATA_DIR.is_dir())


# ============================================================
# 2. CSV 파일 자동 탐색
#    현재 네 폴더에는 통합 데이터 1개가 있음
# ============================================================

all_csv_files = list(DATA_DIR.glob("*.csv"))

print("\n전체 CSV 파일:")
for f in all_csv_files:
    print(f.name)

file_paths = [
    f for f in all_csv_files
    if "통합" in f.name or "2011~2019" in f.name
]

print("\n선택된 파일:")
for f in file_paths:
    print(f.name)

if len(file_paths) == 0:
    raise FileNotFoundError("통합 CSV 파일을 찾지 못했습니다. 파일명에 '통합' 또는 '2011~2019'가 들어있는지 확인하세요.")

if len(file_paths) > 1:
    print("\n주의: 통합 CSV가 여러 개 잡혔습니다. 첫 번째 파일만 사용합니다.")

csv_path = file_paths[0]

print("\n사용할 파일:", csv_path)


# ============================================================
# 3. 한글 CSV 읽기 함수
# ============================================================

def read_korean_csv(path):
    encodings = ["utf-8-sig", "cp949", "euc-kr", "utf-8"]
    
    last_error = None
    
    for enc in encodings:
        try:
            print(f"인코딩 시도: {enc}")
            df = pd.read_csv(path, encoding=enc, low_memory=False)
            print(f"성공 인코딩: {enc}")
            return df
        except Exception as e:
            last_error = e
            print(f"실패 인코딩: {enc}")
    
    raise last_error


# ============================================================
# 4. 컬럼명 정리 함수
# ============================================================

def clean_colname(col):
    col = str(col)
    col = col.replace("\n", "")
    col = col.replace("\r", "")
    col = col.replace("\t", "")
    col = col.replace(" ", "")
    col = col.replace("(", "")
    col = col.replace(")", "")
    col = col.replace("/", "_")
    col = col.replace("-", "_")
    col = col.strip()
    return col


def standardize_columns(df):
    df = df.copy()
    df.columns = [clean_colname(c) for c in df.columns]
    
    rename_map = {
        # 날짜/시간
        "사고일": "사고일자",
        "발생일자": "사고일자",
        "발생시간": "사고시간",
        "시간": "사고시간",
        "월별": "월별구분",
        "시간1단위": "시간단위1시간",
        "시간3단위": "시간단위3시간",
        
        # 조직/노선
        "본부": "본부명",
        "지사": "지사명",
        "사고노선": "노선명",
        "노선": "노선명",
        "사고발생이정": "이정",
        "기종점명": "방향",
        
        # 사고 원인
        "사고원인": "주사고원인",
        "원인": "주사고원인",
        "사고원인구분": "사고원인구분",
        
        # 사고 유형
        "주사고유형": "사고유형",
        "사고유형1차": "사고유형",
        "사고유형2차": "사고유형2차",
        
        # 차량/운전자
        "원인차": "원인차차종",
        "원인차종": "원인차차종",
        "피해차": "피해자차종",
        "피해차종": "피해자차종",
        "원인차운전자연령": "원인차운전자연령구분",
        "피해차운전자연령": "피해차운전자연령구분",
        
        # 도로환경
        "종단구배": "종단경사",
        "도로상태": "노면상태",
        "기상": "날씨",
        
        # 피해자 수
        "사망자수": "사망",
        "중상자수": "중상",
        "경상자수": "경상",
    }
    
    df = df.rename(columns=rename_map)
    
    return df


# ============================================================
# 5. 데이터 읽기
# ============================================================

df = read_korean_csv(csv_path)
df = standardize_columns(df)

df["source_file"] = csv_path.name

print("\n데이터 크기:", df.shape)
print("\n컬럼 목록:")
print(df.columns.tolist())

print("\n상위 5행:")
display(df.head())


# ============================================================
# 6. 사고일자 처리
# ============================================================

if "사고일자" in df.columns:
    df["사고일자"] = pd.to_datetime(df["사고일자"], errors="coerce")
    
    df["accident_year"] = df["사고일자"].dt.year
    df["accident_month"] = df["사고일자"].dt.month
    df["accident_day"] = df["사고일자"].dt.day
    df["accident_weekday_num"] = df["사고일자"].dt.weekday
else:
    print("\n주의: 사고일자 컬럼이 없습니다.")
    df["accident_year"] = np.nan
    df["accident_month"] = np.nan
    df["accident_day"] = np.nan
    df["accident_weekday_num"] = np.nan


# ============================================================
# 7. 사고시간 처리
# ============================================================

def extract_hour(x):
    if pd.isna(x):
        return np.nan
    
    x = str(x).strip()
    
    # 09:30 형태
    if ":" in x:
        try:
            h = int(x.split(":")[0])
            if 0 <= h <= 23:
                return h
            return np.nan
        except:
            return np.nan
    
    # 숫자만 추출
    x = re.sub(r"[^0-9]", "", x)
    
    if x == "":
        return np.nan
    
    # 예: 930 -> 9, 0930 -> 9, 1530 -> 15
    if len(x) >= 3:
        h = int(x[:-2])
    else:
        h = int(x)
    
    if 0 <= h <= 23:
        return h
    
    return np.nan


if "사고시간" in df.columns:
    df["accident_hour"] = df["사고시간"].apply(extract_hour)
else:
    print("\n주의: 사고시간 컬럼이 없습니다.")
    df["accident_hour"] = np.nan


def hour_band(h):
    if pd.isna(h):
        return np.nan
    
    h = int(h)
    
    if 0 <= h <= 5:
        return "새벽"
    elif 6 <= h <= 11:
        return "오전"
    elif 12 <= h <= 17:
        return "오후"
    elif 18 <= h <= 23:
        return "야간"
    else:
        return np.nan


df["accident_time_band"] = df["accident_hour"].apply(hour_band)


# ============================================================
# 8. 사망·중상·경상 기반 타깃 변수 생성
#    injury_occurrence = 1: 인명피해 발생
#    injury_occurrence = 0: 사망·중상·경상 모두 없음
# ============================================================

required_injury_cols = ["사망", "중상", "경상"]

for col in required_injury_cols:
    if col not in df.columns:
        print(f"주의: {col} 컬럼이 없어 0으로 생성합니다.")
        df[col] = 0
    
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

df["injury_total"] = df["사망"] + df["중상"] + df["경상"]

df["injury_occurrence"] = np.where(df["injury_total"] > 0, 1, 0)

print("\n타깃 분포:")
print(df["injury_occurrence"].value_counts())

print("\n타깃 비율:")
print(df["injury_occurrence"].value_counts(normalize=True))


# ============================================================
# 9. 데이터 누수 변수 제거 목록 생성
#    사망·중상·경상으로 y를 만들었으므로 관련 컬럼은 X에서 제거
# ============================================================

leakage_keywords = [
    "사망",
    "중상",
    "경상",
    "부상",
    "injury",
    "피해자수",
    "사상자",
    "인명피해",
    "사고등급",
    "피해내용",
    "사고내용",
    "피해액",
    "시설물피해액",
    "차량피해",
    "안전벨트",
    "에어백",
    "유아시트"
]

leakage_cols = [
    col for col in df.columns
    if any(keyword in str(col) for keyword in leakage_keywords)
]

leakage_cols = [
    c for c in leakage_cols
    if c not in ["injury_occurrence"]
]

print("\n제거할 누수 가능 컬럼:")
print(leakage_cols)


# ============================================================
# 10. 불필요 컬럼 제거 목록
# ============================================================

drop_cols = [
    "injury_occurrence",
    "injury_total",
    "source_file",
    
    # 원본 날짜/시간은 파생변수로 대체
    "사고일자",
    
    # ID성 컬럼
    "사고번호",
    "ID",
    "id",
    "No",
    "NO",
    "번호",
    "순번",
    
    # 차량번호는 식별자에 가까워 제거
    "원인차차량번호",
    "피해차차량번호",
    "차량번호"
]

drop_cols = list(set(drop_cols + leakage_cols))

existing_drop_cols = [c for c in drop_cols if c in df.columns]

print("\n실제로 제거되는 컬럼 수:", len(existing_drop_cols))
print(existing_drop_cols)


# ============================================================
# 11. X, y 생성
# ============================================================

X = df.drop(columns=existing_drop_cols, errors="ignore")
y = df["injury_occurrence"].astype(int)

# 완전히 비어 있는 컬럼 제거
empty_cols = X.columns[X.isna().all()].tolist()
X = X.drop(columns=empty_cols)

print("\n전체 결측 컬럼 제거:")
print(empty_cols)

# 상수 컬럼 제거
constant_cols = [
    col for col in X.columns
    if X[col].nunique(dropna=True) <= 1
]

X = X.drop(columns=constant_cols)

print("\n상수 컬럼 제거:")
print(constant_cols)

print("\n최종 X 크기:", X.shape)
print("최종 y 크기:", y.shape)


# ============================================================
# 12. 수치형 / 범주형 변수 분리
# ============================================================

numeric_cols = X.select_dtypes(
    include=["int64", "float64", "int32", "float32"]
).columns.tolist()

categorical_cols = X.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("\n수치형 변수 수:", len(numeric_cols))
print(numeric_cols)

print("\n범주형 변수 수:", len(categorical_cols))
print(categorical_cols)


# ============================================================
# 13. Train / Test Split
#    불균형 데이터이므로 stratify=y 필수
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nTrain 타깃 분포:")
print(y_train.value_counts())
print(y_train.value_counts(normalize=True))

print("\nTest 타깃 분포:")
print(y_test.value_counts())
print(y_test.value_counts(normalize=True))


# ============================================================
# 14. scale_pos_weight 계산
#    XGBoost에서 클래스 불균형 보정에 사용
# ============================================================

negative_count = int((y_train == 0).sum())
positive_count = int((y_train == 1).sum())

scale_pos_weight = negative_count / positive_count

print("\nnegative_count:", negative_count)
print("positive_count:", positive_count)
print("scale_pos_weight:", scale_pos_weight)


# ============================================================
# 15. 전처리 파이프라인
#    XGBoost는 스케일링 불필요
#    수치형: 중앙값 대체
#    범주형: 최빈값 대체 + One-Hot Encoding
# ============================================================

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols)
    ],
    remainder="drop"
)


# ============================================================
# 16. 전처리 적용
# ============================================================

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("\n전처리 후 X_train shape:", X_train_processed.shape)
print("전처리 후 X_test shape:", X_test_processed.shape)


# ============================================================
# 17. One-Hot Encoding 이후 변수명 복원
#    SHAP 분석을 위해 필요
# ============================================================

encoded_cat_cols = []

if len(categorical_cols) > 0:
    encoded_cat_cols = (
        preprocessor
        .named_transformers_["cat"]
        .named_steps["onehot"]
        .get_feature_names_out(categorical_cols)
        .tolist()
    )

final_feature_names = numeric_cols + encoded_cat_cols

X_train_processed = pd.DataFrame(
    X_train_processed,
    columns=final_feature_names,
    index=X_train.index
)

X_test_processed = pd.DataFrame(
    X_test_processed,
    columns=final_feature_names,
    index=X_test.index
)

print("\n최종 전처리 변수 수:", len(final_feature_names))
display(X_train_processed.head())


# ============================================================
# 18. 전처리 결과 저장
# ============================================================

SAVE_DIR = DATA_DIR / "preprocessed_xgboost_injury_occurrence"
SAVE_DIR.mkdir(exist_ok=True)

X_train_processed.to_csv(
    SAVE_DIR / "X_train_processed.csv",
    index=False,
    encoding="utf-8-sig"
)

X_test_processed.to_csv(
    SAVE_DIR / "X_test_processed.csv",
    index=False,
    encoding="utf-8-sig"
)

y_train.to_csv(
    SAVE_DIR / "y_train.csv",
    index=False,
    encoding="utf-8-sig"
)

y_test.to_csv(
    SAVE_DIR / "y_test.csv",
    index=False,
    encoding="utf-8-sig"
)

pd.Series(
    final_feature_names,
    name="feature_name"
).to_csv(
    SAVE_DIR / "final_feature_names.csv",
    index=False,
    encoding="utf-8-sig"
)

pd.DataFrame({
    "negative_count": [negative_count],
    "positive_count": [positive_count],
    "scale_pos_weight": [scale_pos_weight]
}).to_csv(
    SAVE_DIR / "scale_pos_weight.csv",
    index=False,
    encoding="utf-8-sig"
)

pd.Series(
    X.columns,
    name="original_feature_name"
).to_csv(
    SAVE_DIR / "original_feature_names_before_encoding.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\n전처리 결과 저장 완료")
print("저장 경로:", SAVE_DIR)


# ============================================================
# 19. 최종 확인
# ============================================================

print("\n========== 최종 요약 ==========")
print("원본 데이터 크기:", df.shape)
print("전처리 전 X 크기:", X.shape)
print("전처리 후 X_train 크기:", X_train_processed.shape)
print("전처리 후 X_test 크기:", X_test_processed.shape)
print("y_train 크기:", y_train.shape)
print("y_test 크기:", y_test.shape)
print("scale_pos_weight:", scale_pos_weight)

print("\n타깃 분포 전체:")
print(y.value_counts())
print(y.value_counts(normalize=True))

In [ ]:
# ============================================================
# 저장된 전처리 데이터 다시 불러오기
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

# 원본 데이터 폴더
DATA_DIR = Path(
    "/Users/leeseungbo/Desktop/APTE_교통사고_데이터/DATA/2011~2019_고속도로 교통사고 데이터"
)

# 전처리 결과 저장 폴더
PREPROCESSED_DIR = DATA_DIR / "preprocessed_xgboost_injury_occurrence"

print("전처리 폴더 존재 여부:", PREPROCESSED_DIR.exists())
print("전처리 폴더 경로:", PREPROCESSED_DIR)

print("\n폴더 안 파일:")
for f in PREPROCESSED_DIR.glob("*"):
    print(f.name)


# ============================================================
# CSV 불러오기
# ============================================================

X_train_processed = pd.read_csv(
    PREPROCESSED_DIR / "X_train_processed.csv",
    encoding="utf-8-sig"
)

X_test_processed = pd.read_csv(
    PREPROCESSED_DIR / "X_test_processed.csv",
    encoding="utf-8-sig"
)

y_train = pd.read_csv(
    PREPROCESSED_DIR / "y_train.csv",
    encoding="utf-8-sig"
).iloc[:, 0]

y_test = pd.read_csv(
    PREPROCESSED_DIR / "y_test.csv",
    encoding="utf-8-sig"
).iloc[:, 0]

scale_pos_weight_df = pd.read_csv(
    PREPROCESSED_DIR / "scale_pos_weight.csv",
    encoding="utf-8-sig"
)

scale_pos_weight = float(scale_pos_weight_df["scale_pos_weight"].iloc[0])


# ============================================================
# 불러오기 확인
# ============================================================

print("\n불러오기 완료")
print("X_train_processed shape:", X_train_processed.shape)
print("X_test_processed shape:", X_test_processed.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)
print("scale_pos_weight:", scale_pos_weight)

print("\ny_train 분포:")
print(y_train.value_counts())
print(y_train.value_counts(normalize=True))

print("\ny_test 분포:")
print(y_test.value_counts())
print(y_test.value_counts(normalize=True))

In [ ]:
# ============================================================
# 클래스 불균형 시각화
# 기준 변수: injury_occurrence
# 0 = 인명피해 없음
# 1 = 인명피해 발생
# ============================================================

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# 한글 폰트 설정 - Mac 기준
plt.rcParams["font.family"] = "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False

# ============================================================
# y_train + y_test 합쳐서 전체 타깃 분포 확인
# ============================================================

target_col = "injury_occurrence"

y_all = pd.concat([y_train, y_test], axis=0).reset_index(drop=True)

class_counts = y_all.value_counts().sort_index()
class_ratios = y_all.value_counts(normalize=True).sort_index() * 100

class_summary = pd.DataFrame({
    "class": class_counts.index,
    "count": class_counts.values,
    "ratio_percent": class_ratios.values
})

class_summary["label"] = class_summary["class"].map({
    0: "인명피해 없음",
    1: "인명피해 발생"
})

print(class_summary)


# ============================================================
# 1. 클래스별 건수 시각화
# ============================================================

plt.figure(figsize=(7, 5))

bars = plt.bar(
    class_summary["label"],
    class_summary["count"]
)

plt.title("Class Imbalance: Injury Occurrence Count", fontsize=14)
plt.xlabel("Class", fontsize=12)
plt.ylabel("Count", fontsize=12)

# 막대 위에 건수와 비율 표시
for bar, count, ratio in zip(
    bars,
    class_summary["count"],
    class_summary["ratio_percent"]
):
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        height,
        f"{int(count):,}\n({ratio:.2f}%)",
        ha="center",
        va="bottom",
        fontsize=11
    )

plt.tight_layout()
plt.show()

# ============================================================
# Train/Test 클래스 분포 비교 시각화
# ============================================================

train_summary = y_train.value_counts().sort_index()
test_summary = y_test.value_counts().sort_index()

compare_df = pd.DataFrame({
    "Train": train_summary,
    "Test": test_summary
}).fillna(0).astype(int)

compare_df.index = compare_df.index.map({
    0: "인명피해 없음",
    1: "인명피해 발생"
})

print(compare_df)

ax = compare_df.plot(
    kind="bar",
    figsize=(8, 5)
)

plt.title("Train/Test Class Distribution: Injury Occurrence", fontsize=14)
plt.xlabel("Class", fontsize=12)
plt.ylabel("Count", fontsize=12)
plt.xticks(rotation=0)

# 막대 위 숫자 표시
for container in ax.containers:
    ax.bar_label(
        container,
        fmt="%.0f",
        label_type="edge",
        fontsize=10
    )

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# XGBoost 이진분류 전체 코드
# injury_occurrence 예측
# - scale_pos_weight 기반 클래스 불균형 보정
# - 기본 threshold 0.5 성능평가
# - F1-score 기준 최적 threshold 탐색
# - Confusion Matrix, ROC Curve, PR Curve 시각화
# - 결과 저장
# ============================================================


# ============================================================
# 0. 라이브러리
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    precision_recall_curve
)


# ============================================================
# 1. 한글 폰트 설정 - Mac 기준
# ============================================================

plt.rcParams["font.family"] = "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False


# ============================================================
# 2. 결과 저장 경로
# ============================================================

SAVE_DIR = DATA_DIR / "xgboost_injury_occurrence_results"
SAVE_DIR.mkdir(exist_ok=True)

print("결과 저장 경로:", SAVE_DIR)


# ============================================================
# 3. 데이터 확인
# ============================================================

print("X_train_processed shape:", X_train_processed.shape)
print("X_test_processed shape:", X_test_processed.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

print("\ny_train 분포:")
print(y_train.value_counts())
print(y_train.value_counts(normalize=True))

print("\ny_test 분포:")
print(y_test.value_counts())
print(y_test.value_counts(normalize=True))

print("\nscale_pos_weight:", scale_pos_weight)


# ============================================================
# 4. XGBoost 모델 정의
# ============================================================

xgb_model = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",

    n_estimators=500,
    max_depth=4,
    learning_rate=0.03,

    subsample=0.8,
    colsample_bytree=0.8,

    min_child_weight=3,
    gamma=0.1,

    reg_alpha=0.1,
    reg_lambda=1.0,

    scale_pos_weight=scale_pos_weight,

    random_state=42,
    n_jobs=-1,
    tree_method="hist"
)


# ============================================================
# 5. 모델 학습
# ============================================================

xgb_model.fit(
    X_train_processed,
    y_train
)

print("\nXGBoost 학습 완료")


# ============================================================
# 6. 예측 확률 산출
# ============================================================

y_train_proba = xgb_model.predict_proba(X_train_processed)[:, 1]
y_test_proba = xgb_model.predict_proba(X_test_processed)[:, 1]

print("\n예측 확률 산출 완료")
print("y_test_proba 최소값:", y_test_proba.min())
print("y_test_proba 최대값:", y_test_proba.max())
print("y_test_proba 평균값:", y_test_proba.mean())


# ============================================================
# 7. 성능평가 함수
# ============================================================

def evaluate_binary_classification(y_true, y_pred, y_proba, dataset_name="Test"):
    result = {
        "dataset": dataset_name,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1_score": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_proba),
        "pr_auc": average_precision_score(y_true, y_proba)
    }
    
    return result


# ============================================================
# 8. 기본 threshold 0.5 적용
# ============================================================

default_threshold = 0.5

y_train_pred_default = (y_train_proba >= default_threshold).astype(int)
y_test_pred_default = (y_test_proba >= default_threshold).astype(int)

train_result_default = evaluate_binary_classification(
    y_train,
    y_train_pred_default,
    y_train_proba,
    dataset_name="Train_threshold_0.5"
)

test_result_default = evaluate_binary_classification(
    y_test,
    y_test_pred_default,
    y_test_proba,
    dataset_name="Test_threshold_0.5"
)

default_result_df = pd.DataFrame([
    train_result_default,
    test_result_default
])

print("\n========== 기본 threshold 0.5 성능 ==========")
display(default_result_df)

print("\nTest Classification Report - threshold 0.5")
print(classification_report(y_test, y_test_pred_default, digits=4))


# ============================================================
# 9. Confusion Matrix 시각화 함수
# ============================================================

def plot_confusion_matrix(y_true, y_pred, title, save_path=None):
    cm = confusion_matrix(y_true, y_pred)

    plt.figure(figsize=(6, 5))
    plt.imshow(cm)
    plt.title(title, fontsize=14)
    plt.xlabel("Predicted label")
    plt.ylabel("True label")

    plt.xticks([0, 1], ["No injury", "Injury"])
    plt.yticks([0, 1], ["No injury", "Injury"])

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(
                j,
                i,
                f"{cm[i, j]:,}",
                ha="center",
                va="center",
                fontsize=13
            )

    plt.colorbar()
    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")

    plt.show()

    return cm


# ============================================================
# 10. 기본 threshold Confusion Matrix
# ============================================================

cm_default = plot_confusion_matrix(
    y_test,
    y_test_pred_default,
    title="Confusion Matrix - XGBoost Threshold 0.5",
    save_path=SAVE_DIR / "confusion_matrix_threshold_0_5.png"
)

print("\nConfusion Matrix - threshold 0.5")
print(cm_default)


# ============================================================
# 11. ROC Curve
# ============================================================

fpr, tpr, roc_thresholds = roc_curve(y_test, y_test_proba)
test_roc_auc = roc_auc_score(y_test, y_test_proba)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f"ROC AUC = {test_roc_auc:.4f}")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.title("ROC Curve - XGBoost", fontsize=14)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.tight_layout()
plt.savefig(SAVE_DIR / "roc_curve_xgboost.png", dpi=300, bbox_inches="tight")
plt.show()


# ============================================================
# 12. Precision-Recall Curve
# ============================================================

precision_curve, recall_curve, pr_thresholds = precision_recall_curve(
    y_test,
    y_test_proba
)

test_pr_auc = average_precision_score(y_test, y_test_proba)

plt.figure(figsize=(7, 5))
plt.plot(recall_curve, precision_curve, label=f"PR AUC = {test_pr_auc:.4f}")
plt.title("Precision-Recall Curve - XGBoost", fontsize=14)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.legend()
plt.tight_layout()
plt.savefig(SAVE_DIR / "pr_curve_xgboost.png", dpi=300, bbox_inches="tight")
plt.show()


# ============================================================
# 13. Threshold별 성능 계산
# ============================================================

thresholds = np.arange(0.05, 0.95, 0.01)

threshold_results = []

for threshold in thresholds:
    y_pred = (y_test_proba >= threshold).astype(int)

    result = {
        "threshold": threshold,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1_score": f1_score(y_test, y_pred, zero_division=0),
    }

    threshold_results.append(result)

threshold_df = pd.DataFrame(threshold_results)

print("\n========== Threshold별 성능 일부 ==========")
display(threshold_df.head())


# ============================================================
# 14. F1-score 기준 최적 threshold 선택
# ============================================================

best_f1_row = threshold_df.loc[threshold_df["f1_score"].idxmax()]
best_threshold = float(best_f1_row["threshold"])

print("\n========== F1 기준 최적 threshold ==========")
print("Best threshold:", round(best_threshold, 3))
display(best_f1_row.to_frame().T)


# ============================================================
# 15. Recall 기준 참고 threshold
#    필요하면 발표에서 비교용으로 사용
# ============================================================

# recall이 0.7 이상인 threshold 중 f1이 가장 높은 값
recall_target = 0.70

recall_candidate_df = threshold_df[threshold_df["recall"] >= recall_target].copy()

if len(recall_candidate_df) > 0:
    best_recall_row = recall_candidate_df.loc[recall_candidate_df["f1_score"].idxmax()]
    recall_based_threshold = float(best_recall_row["threshold"])

    print(f"\n========== Recall {recall_target} 이상 조건에서 F1 최적 threshold ==========")
    print("Recall-based threshold:", round(recall_based_threshold, 3))
    display(best_recall_row.to_frame().T)
else:
    recall_based_threshold = None
    print(f"\nRecall {recall_target} 이상을 만족하는 threshold가 없습니다.")


# ============================================================
# 16. 최적 threshold 적용
# ============================================================

y_train_pred_best = (y_train_proba >= best_threshold).astype(int)
y_test_pred_best = (y_test_proba >= best_threshold).astype(int)

train_result_best = evaluate_binary_classification(
    y_train,
    y_train_pred_best,
    y_train_proba,
    dataset_name=f"Train_threshold_{best_threshold:.2f}"
)

test_result_best = evaluate_binary_classification(
    y_test,
    y_test_pred_best,
    y_test_proba,
    dataset_name=f"Test_threshold_{best_threshold:.2f}"
)

best_result_df = pd.DataFrame([
    train_result_best,
    test_result_best
])

print("\n========== 최적 threshold 성능 ==========")
display(best_result_df)

print("\nTest Classification Report - Best F1 Threshold")
print(classification_report(y_test, y_test_pred_best, digits=4))


# ============================================================
# 17. Threshold별 Precision / Recall / F1 시각화
# ============================================================

plt.figure(figsize=(8, 5))

plt.plot(threshold_df["threshold"], threshold_df["precision"], label="Precision")
plt.plot(threshold_df["threshold"], threshold_df["recall"], label="Recall")
plt.plot(threshold_df["threshold"], threshold_df["f1_score"], label="F1-score")

plt.axvline(
    best_threshold,
    linestyle="--",
    label=f"Best F1 threshold = {best_threshold:.2f}"
)

plt.title("Performance by Classification Threshold", fontsize=14)
plt.xlabel("Threshold")
plt.ylabel("Score")
plt.legend()
plt.tight_layout()
plt.savefig(SAVE_DIR / "threshold_performance_curve.png", dpi=300, bbox_inches="tight")
plt.show()


# ============================================================
# 18. 최적 threshold Confusion Matrix
# ============================================================

cm_best = plot_confusion_matrix(
    y_test,
    y_test_pred_best,
    title=f"Confusion Matrix - XGBoost Threshold {best_threshold:.2f}",
    save_path=SAVE_DIR / "confusion_matrix_best_threshold.png"
)

print("\nConfusion Matrix - best threshold")
print(cm_best)


# ============================================================
# 19. 예측 확률 분포 시각화
# ============================================================

plt.figure(figsize=(8, 5))

plt.hist(
    y_test_proba[y_test.values == 0],
    bins=50,
    alpha=0.6,
    label="No injury"
)

plt.hist(
    y_test_proba[y_test.values == 1],
    bins=50,
    alpha=0.6,
    label="Injury"
)

plt.axvline(
    best_threshold,
    linestyle="--",
    label=f"Best threshold = {best_threshold:.2f}"
)

plt.title("Predicted Probability Distribution by True Class", fontsize=14)
plt.xlabel("Predicted probability of injury occurrence")
plt.ylabel("Frequency")
plt.legend()
plt.tight_layout()
plt.savefig(SAVE_DIR / "predicted_probability_distribution.png", dpi=300, bbox_inches="tight")
plt.show()


# ============================================================
# 20. Feature Importance - XGBoost 기본 중요도
#     SHAP 전 단계 참고용
# ============================================================

feature_importance_df = pd.DataFrame({
    "feature": X_train_processed.columns,
    "importance": xgb_model.feature_importances_
})

feature_importance_df = feature_importance_df.sort_values(
    by="importance",
    ascending=False
)

print("\n========== XGBoost 기본 Feature Importance Top 30 ==========")
display(feature_importance_df.head(30))

plt.figure(figsize=(8, 10))

top_n = 30
top_features = feature_importance_df.head(top_n).sort_values(
    by="importance",
    ascending=True
)

plt.barh(
    top_features["feature"],
    top_features["importance"]
)

plt.title(f"XGBoost Feature Importance Top {top_n}", fontsize=14)
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.savefig(SAVE_DIR / "xgboost_feature_importance_top30.png", dpi=300, bbox_inches="tight")
plt.show()


# ============================================================
# 21. 결과 저장
# ============================================================

all_result_df = pd.concat(
    [
        default_result_df,
        best_result_df
    ],
    ignore_index=True
)

all_result_df.to_csv(
    SAVE_DIR / "xgboost_performance_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

threshold_df.to_csv(
    SAVE_DIR / "threshold_search_results.csv",
    index=False,
    encoding="utf-8-sig"
)

feature_importance_df.to_csv(
    SAVE_DIR / "xgboost_feature_importance.csv",
    index=False,
    encoding="utf-8-sig"
)

prediction_result_df = pd.DataFrame({
    "y_true": y_test.values,
    "y_proba": y_test_proba,
    "y_pred_threshold_0_5": y_test_pred_default,
    "y_pred_best_threshold": y_test_pred_best
})

prediction_result_df.to_csv(
    SAVE_DIR / "xgboost_prediction_results.csv",
    index=False,
    encoding="utf-8-sig"
)

# Confusion Matrix 저장
cm_default_df = pd.DataFrame(
    cm_default,
    index=["Actual_No_Injury", "Actual_Injury"],
    columns=["Pred_No_Injury", "Pred_Injury"]
)

cm_best_df = pd.DataFrame(
    cm_best,
    index=["Actual_No_Injury", "Actual_Injury"],
    columns=["Pred_No_Injury", "Pred_Injury"]
)

cm_default_df.to_csv(
    SAVE_DIR / "confusion_matrix_threshold_0_5.csv",
    encoding="utf-8-sig"
)

cm_best_df.to_csv(
    SAVE_DIR / "confusion_matrix_best_threshold.csv",
    encoding="utf-8-sig"
)


# ============================================================
# 22. 논문용 성능표 생성
# ============================================================

paper_result_df = all_result_df.copy()

metric_cols = [
    "accuracy",
    "precision",
    "recall",
    "f1_score",
    "roc_auc",
    "pr_auc"
]

for col in metric_cols:
    paper_result_df[col] = paper_result_df[col].round(4)

paper_result_df.columns = [
    "Dataset",
    "Accuracy",
    "Precision",
    "Recall",
    "F1-score",
    "ROC-AUC",
    "PR-AUC"
]

paper_result_df.to_csv(
    SAVE_DIR / "paper_xgboost_performance_table.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\n========== 논문용 성능표 ==========")
display(paper_result_df)


# ============================================================
# 23. 최종 요약 출력
# ============================================================

print("\n========== 최종 요약 ==========")
print("모델: XGBoost Binary Classification")
print("Target: injury_occurrence")
print("Class imbalance method: scale_pos_weight")
print("scale_pos_weight:", round(scale_pos_weight, 4))
print("Default threshold:", default_threshold)
print("Best F1 threshold:", round(best_threshold, 4))

print("\nTest 성능 - threshold 0.5")
print("Accuracy :", round(test_result_default["accuracy"], 4))
print("Precision:", round(test_result_default["precision"], 4))
print("Recall   :", round(test_result_default["recall"], 4))
print("F1-score :", round(test_result_default["f1_score"], 4))
print("ROC-AUC  :", round(test_result_default["roc_auc"], 4))
print("PR-AUC   :", round(test_result_default["pr_auc"], 4))

print("\nTest 성능 - Best F1 threshold")
print("Accuracy :", round(test_result_best["accuracy"], 4))
print("Precision:", round(test_result_best["precision"], 4))
print("Recall   :", round(test_result_best["recall"], 4))
print("F1-score :", round(test_result_best["f1_score"], 4))
print("ROC-AUC  :", round(test_result_best["roc_auc"], 4))
print("PR-AUC   :", round(test_result_best["pr_auc"], 4))

print("\n결과 저장 완료:")
print(SAVE_DIR)